# HPI-GCN-RP: Violence Detection (All-in-One Kaggle Notebook)

**Architecture:** ST-GCN graph neural network with inter-person interaction edges and adaptive relation prediction.

This notebook does **everything on Kaggle** — extraction + training + export:
1. Extracts skeletons from NTU RGB+D `.skeleton` files
2. Trains the HPI-GCN-RP model (50 epochs, GPU)
3. Exports `hpi_gcn_rp.pt` for download

---

### How to use (step by step):

**Step 1 — Get NTU RGB+D skeleton data:**
- Download from Google Drive (official): `1CUZnBtYwifVXS21yVg62T-vrPVayso5H` (search "shahroudy NTURGB-D github" for links)
- Or request from ROSE Lab: https://rose1.ntu.edu.sg/dataset/actionRecognition/
- You only need the skeleton files (~600 MB), NOT the RGB videos

**Step 2 — Upload to Kaggle as dataset:**
1. Open https://kaggle.com/datasets
2. Click **+ New Dataset** (blue button)
3. Name it anything you want (e.g., `ntu-skeletons`)
4. Upload the downloaded zip file (don't unzip — Kaggle does it automatically)
5. Click **Create**

**Step 3 — Create Kaggle Notebook:**
1. Open https://kaggle.com/code
2. Click **+ New Notebook** (blue button)

**Step 4 — Add dataset + import notebook:**
1. On the right panel click **+ Add Input** -> find your dataset -> **Add**
2. Click **File -> Import Notebook** -> upload this `.ipynb` file
3. (No need to edit any paths — the notebook auto-detects your data!)

**Step 5 — Enable GPU:**
1. On the right: **Settings** (gear) -> **Accelerator** -> **GPU P100** or **T4 x2**
2. **Persistence** -> **Files only**
3. **Internet** -> **On**

**Step 6 — Run:**
1. Click **Run All** (or Ctrl+F9)
2. Skeleton extraction (~3-5 min) -> Training (~15-30 min) -> Done!

**Step 7 — Download the model:**
1. On the left panel click **Output**
2. Find `hpi_gcn_rp.pt` -> **Download**
3. Rename to `best_model.pth`
4. Put in `AiGuardian/models/best_model.pth`

## 0. Skeleton Extraction from NTU RGB+D

This section extracts and converts raw NTU `.skeleton` files into training-ready `.npy` arrays.
- **Input:** NTU RGB+D `.skeleton` files (actions A050-A060)
- **Output:** `(30, 102)` numpy arrays in `fight/` and `nonfight/` folders
- No YOLO or GPU needed — pure CPU parsing

In [ ]:
import json
import time as _time
import numpy as np
from pathlib import Path

# ── Extraction Config ──
SEQUENCE_LEN = 30        # frames per clip
MAX_PEOPLE = 2            # max persons per frame
NUM_KEYPOINTS = 17        # COCO keypoints
FEATURES_PER_KP = 3       # x_norm, y_norm, confidence
FEATURE_DIM = MAX_PEOPLE * NUM_KEYPOINTS * FEATURES_PER_KP  # 102

# NTU 25-joint -> COCO 17-joint mapping
NTU_TO_COCO = {
    0: 3, 1: 3, 2: 3, 3: 3, 4: 3,  # head joints -> nose approx
    5: 4,   # l_shoulder
    6: 8,   # r_shoulder
    7: 5,   # l_elbow
    8: 9,   # r_elbow
    9: 6,   # l_wrist
    10: 10, # r_wrist
    11: 12, # l_hip
    12: 16, # r_hip
    13: 13, # l_knee
    14: 17, # r_knee
    15: 14, # l_ankle
    16: 18, # r_ankle
}

# NTU action classes for violence detection
NTU_FIGHT_CLASSES = {50, 51, 52}        # punch/slap, kicking, pushing
NTU_NORMAL_CLASSES = {53, 54, 55, 56, 57, 58, 59, 60}  # hard negatives


def parse_ntu_skeleton(filepath):
    """Parse NTU RGB+D .skeleton file -> list of frames, each = list of bodies (25,3)."""
    frames = []
    with open(filepath, 'r') as f:
        num_frames = int(f.readline().strip())
        for _ in range(num_frames):
            num_bodies = int(f.readline().strip())
            bodies = []
            for _ in range(num_bodies):
                _ = f.readline()  # body info (skip)
                num_joints = int(f.readline().strip())
                joints = np.zeros((num_joints, 3), dtype=np.float32)
                for j in range(num_joints):
                    parts = f.readline().strip().split()
                    joints[j] = [float(parts[0]), float(parts[1]), float(parts[2])]
                bodies.append(joints)
            frames.append(bodies)
    return frames


def ntu_to_coco_skeleton(ntu_joints_25):
    """Convert NTU 25-joint (x,y,z meters) -> COCO 17-joint (x_norm, y_norm, conf=1.0)."""
    coco = np.zeros((17, 3), dtype=np.float32)
    for coco_idx, ntu_idx in NTU_TO_COCO.items():
        pt = ntu_joints_25[ntu_idx]
        coco[coco_idx, 0] = (pt[0] + 1.5) / 3.0  # x normalized
        coco[coco_idx, 1] = (pt[1] + 0.5) / 2.0  # y normalized
        coco[coco_idx, 2] = 1.0                    # always confident
    return coco


def get_ntu_action_class(filename):
    """Extract action class from NTU filename: SsssCcccPpppRrrrAaaa."""
    try:
        name = Path(filename).stem
        a_idx = name.find('A')
        if a_idx >= 0:
            return int(name[a_idx+1:a_idx+4])
    except:
        pass
    return -1


def process_ntu_dataset(input_dir, output_dir):
    """Extract NTU .skeleton files -> (30, 102) numpy arrays for training."""
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    stats = {"total": 0, "processed": 0, "failed": 0, "skipped_class": 0}

    skeleton_files = sorted(input_path.rglob("*.skeleton"))
    print(f"Found {len(skeleton_files)} .skeleton files in {input_dir}")

    if len(skeleton_files) == 0:
        print("ERROR: No .skeleton files found!")
        print(f"Check that INPUT_DIR points to the correct path.")
        print(f"Expected: /kaggle/input/<your-dataset-name>/...")
        return

    start = _time.time()

    for i, sf in enumerate(skeleton_files):
        action = get_ntu_action_class(sf.name)

        if action in NTU_FIGHT_CLASSES:
            label = "fight"
        elif action in NTU_NORMAL_CLASSES:
            label = "nonfight"
        else:
            stats["skipped_class"] += 1
            continue

        stats["total"] += 1
        out_dir = output_path / "train" / label
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{sf.stem}.npy"

        if out_file.exists():
            stats["processed"] += 1
            continue

        try:
            frames = parse_ntu_skeleton(str(sf))
            if len(frames) < 5:
                stats["failed"] += 1
                continue

            indices = np.linspace(0, len(frames) - 1, SEQUENCE_LEN, dtype=int)
            all_features = []

            for idx in indices:
                bodies = frames[idx]
                frame_kps = []

                for body in bodies[:MAX_PEOPLE]:
                    if len(body) >= 25:
                        coco_kp = ntu_to_coco_skeleton(body)
                        frame_kps.append(coco_kp.flatten())

                while len(frame_kps) < MAX_PEOPLE:
                    frame_kps.append(np.zeros(NUM_KEYPOINTS * FEATURES_PER_KP, dtype=np.float32))
                frame_kps = frame_kps[:MAX_PEOPLE]

                all_features.append(np.concatenate(frame_kps))

            result = np.array(all_features, dtype=np.float32)
            np.save(str(out_file), result)
            stats["processed"] += 1

        except Exception as e:
            stats["failed"] += 1
            if i < 5:
                print(f"  Error: {sf.name}: {e}")

        if (i + 1) % 200 == 0:
            elapsed = _time.time() - start
            print(f"  [{i+1}/{len(skeleton_files)}] "
                  f"ok={stats['processed']} fail={stats['failed']} "
                  f"skip={stats['skipped_class']} ({elapsed:.0f}s)")

    elapsed = _time.time() - start
    print(f"\n{'='*50}")
    print(f"EXTRACTION DONE: {stats['processed']}/{stats['total']} processed")
    print(f"Failed: {stats['failed']}, Skipped (wrong class): {stats['skipped_class']}")
    print(f"Feature shape: ({SEQUENCE_LEN}, {FEATURE_DIM})")
    print(f"Output: {output_path}")
    print(f"Time: {elapsed/60:.1f} min")

    # Save metadata
    meta = output_path / "metadata.json"
    with open(str(meta), "w") as f:
        json.dump(stats, f, indent=2)

print("Extraction functions loaded.")

In [ ]:
import os
from pathlib import Path

OUTPUT_DIR = "/kaggle/working/skeletons/ntu"

# ══════════════════════════════════════════════════════════════
# AUTO-DETECT: find .skeleton files anywhere in /kaggle/input/
# No need to edit anything — this cell finds the path for you.
# ══════════════════════════════════════════════════════════════

INPUT_DIR = None

if os.path.exists("/kaggle/input"):
    # Find first .skeleton file and use its parent directory
    for skeleton_file in Path("/kaggle/input").rglob("*.skeleton"):
        INPUT_DIR = str(skeleton_file.parent)
        print(f"AUTO-DETECTED: Found .skeleton files in:\n  {INPUT_DIR}")
        break

if INPUT_DIR is None:
    print("AUTO-DETECT FAILED: No .skeleton files found in /kaggle/input/")
    print()
    print("Available input directories:")
    if os.path.exists("/kaggle/input"):
        for d in sorted(os.listdir("/kaggle/input")):
            full = f"/kaggle/input/{d}"
            if os.path.isdir(full):
                # Count files in top 2 levels
                file_count = sum(1 for _ in Path(full).rglob("*") if _.is_file())
                # Show some example filenames
                examples = [f.name for f in Path(full).rglob("*") if f.is_file()][:3]
                print(f"  {full}/  ({file_count} files)")
                for ex in examples:
                    print(f"    e.g.: {ex}")
    else:
        print("  /kaggle/input/ does not exist! Did you add a dataset?")
    print()
    print("FIX: Set INPUT_DIR manually below, then re-run this cell:")
    INPUT_DIR = "/kaggle/input/CHANGE_ME"
    raise RuntimeError("Cannot find .skeleton files. See instructions above.")

# Count skeleton files to confirm
skeleton_files = list(Path(INPUT_DIR).rglob("*.skeleton"))
print(f"Total .skeleton files: {len(skeleton_files)}")
if len(skeleton_files) > 0:
    print(f"Example: {skeleton_files[0].name}")

# Run extraction
print(f"\nExtracting to: {OUTPUT_DIR}")
process_ntu_dataset(INPUT_DIR, OUTPUT_DIR)

In [ ]:
# Verify extraction results
from pathlib import Path

output = Path(OUTPUT_DIR)
if not output.exists():
    print("ERROR: Extraction did not produce any output.")
    print("Check the errors in the cell above.")
    print(f"Expected output directory: {OUTPUT_DIR}")
else:
    total_files = 0
    for split in sorted(output.iterdir()):
        if not split.is_dir():
            continue
        for cls_dir in sorted(split.iterdir()):
            if not cls_dir.is_dir():
                continue
            files = list(cls_dir.glob("*.npy"))
            total_files += len(files)
            if files:
                sample = np.load(str(files[0]))
                print(f"  {split.name}/{cls_dir.name}: {len(files)} files, shape={sample.shape}")

    if total_files == 0:
        print("WARNING: No .npy files extracted. Check cell above for errors.")
    else:
        print(f"\nTotal extracted: {total_files} files")
        print(f"SKELETON_DIR for training: {OUTPUT_DIR}")
        print("Next cells will train the model.")

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import deque
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'Memory: {mem / 1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ══════════════════════════════════════════════
# Points to extracted skeletons from Section 0
# ══════════════════════════════════════════════
SKELETON_DIR = "/kaggle/working/skeletons/ntu"

# If running locally with pre-extracted data, uncomment:
# SKELETON_DIR = "./skeletons/ntu"

# ── Hyperparameters ──
SEQUENCE_LEN = 30       # frames per clip
NUM_KEYPOINTS = 17      # COCO keypoints
NUM_PEOPLE = 2          # max persons per frame
IN_CHANNELS = 3         # x, y, confidence
NUM_NODES = NUM_KEYPOINTS * NUM_PEOPLE  # 34
FEATURE_DIM = NUM_NODES * IN_CHANNELS   # 102

BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 50
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.05

# NTU cross-subject split
TRAIN_SUBJECTS = {1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38}

print(f"Skeleton dir: {SKELETON_DIR}")
print(f"Exists: {os.path.exists(SKELETON_DIR)}")

## 2. Graph Construction

In [ ]:
def build_coco_adjacency(num_keypoints=17, num_people=2):
    """
    Build 34x34 adjacency matrix for 2-person COCO skeletons.
    Returns: (3, V, V) numpy array — [A_self, A_inward, A_outward].
    """
    V = num_keypoints * num_people  # 34

    # COCO bone connections (undirected)
    intra_edges = [
        (0, 1), (0, 2), (1, 3), (2, 4),       # head
        (5, 6),                                  # shoulders
        (5, 7), (7, 9),                          # left arm
        (6, 8), (8, 10),                         # right arm
        (5, 11), (6, 12),                        # torso sides
        (11, 12),                                # hips
        (11, 13), (13, 15),                      # left leg
        (12, 14), (14, 16),                      # right leg
    ]

    # Full edge list
    edges = []
    for i, j in intra_edges:
        edges.append((i, j))           # Person 1
    for i, j in intra_edges:
        edges.append((i + 17, j + 17)) # Person 2
    for i in range(17):
        edges.append((i, i + 17))      # Inter-person

    # BFS distance from center nodes (nose of each person)
    adj = {i: set() for i in range(V)}
    for i, j in edges:
        adj[i].add(j)
        adj[j].add(i)

    dist = [float('inf')] * V
    queue = deque()
    for c in [0, 17]:  # nose of P1 and P2
        dist[c] = 0
        queue.append(c)
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if dist[v] > dist[u] + 1:
                dist[v] = dist[u] + 1
                queue.append(v)

    # Partition: self, inward (toward center), outward (away from center)
    A_self = np.eye(V, dtype=np.float32)
    A_inward = np.zeros((V, V), dtype=np.float32)
    A_outward = np.zeros((V, V), dtype=np.float32)

    for i, j in edges:
        if dist[i] < dist[j]:
            A_inward[j][i] = 1
            A_outward[i][j] = 1
        elif dist[j] < dist[i]:
            A_inward[i][j] = 1
            A_outward[j][i] = 1
        else:
            A_inward[i][j] = 1
            A_inward[j][i] = 1

    # Normalize: D^{-1/2} A D^{-1/2}
    def _normalize(A_part):
        D = np.sum(A_part, axis=1)
        D_inv_sqrt = np.diag(np.where(D > 0, 1.0 / np.sqrt(D), 0))
        return D_inv_sqrt @ A_part @ D_inv_sqrt

    A = np.stack([_normalize(A_self), _normalize(A_inward), _normalize(A_outward)])
    print(f"Adjacency matrix: {A.shape}")
    print(f"Total edges: {len(edges)} (intra: {len(intra_edges)*2}, inter: 17)")
    return A

# Build and visualize
A_partitions = build_coco_adjacency()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ['A_self (identity)', 'A_inward (toward center)', 'A_outward (away from center)']
for i, (ax, title) in enumerate(zip(axes, titles)):
    im = ax.imshow(A_partitions[i], cmap='hot', interpolation='nearest')
    ax.set_title(title)
    ax.set_xlabel('Node')
    ax.set_ylabel('Node')
    ax.axhline(16.5, color='cyan', linewidth=0.5, linestyle='--')
    ax.axvline(16.5, color='cyan', linewidth=0.5, linestyle='--')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('HPI-GCN-RP: Adjacency Matrix Partitions (34x34)', fontsize=14)
plt.tight_layout()
plt.savefig('adjacency_matrix.png', dpi=150)
plt.show()

## 3. Model Architecture

In [ ]:
class SpatialGraphConv(nn.Module):
    """Graph convolution with spatial partitioning (3 sub-adjacency matrices)."""

    def __init__(self, in_channels, out_channels, A, num_partitions=3):
        super().__init__()
        self.num_partitions = num_partitions
        self.register_buffer('A', torch.tensor(A, dtype=torch.float32))
        self.conv = nn.Conv2d(in_channels, out_channels * num_partitions, kernel_size=1)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        B, C, T, V = x.shape
        x_conv = self.conv(x)
        C_out = x_conv.shape[1] // self.num_partitions
        x_conv = x_conv.reshape(B, self.num_partitions, C_out, T, V)

        out = torch.zeros(B, C_out, T, V, device=x.device)
        for k in range(self.num_partitions):
            out = out + torch.einsum('bctv,vw->bctw', x_conv[:, k], self.A[k])

        return self.bn(out)


class RelationPrediction(nn.Module):
    """
    RP module — learns adaptive inter-person interaction edges.
    B: globally-learned bias (which joints tend to interact)
    Attention: data-dependent (which joints interact in THIS sample)
    """

    def __init__(self, in_channels, num_nodes=34):
        super().__init__()
        self.B = nn.Parameter(torch.zeros(num_nodes, num_nodes))
        self.W_q = nn.Conv2d(in_channels, max(in_channels // 4, 1), kernel_size=1)
        self.W_k = nn.Conv2d(in_channels, max(in_channels // 4, 1), kernel_size=1)

    def forward(self, x):
        B, C, T, V = x.shape
        x_pool = x.mean(dim=2, keepdim=True)
        q = self.W_q(x_pool).squeeze(2)
        k = self.W_k(x_pool).squeeze(2)
        d = q.shape[1]
        attention = torch.bmm(q.permute(0, 2, 1), k) / (d ** 0.5)
        attention = F.softmax(attention, dim=-1)
        C_adaptive = attention + self.B.unsqueeze(0)
        return torch.einsum('bctv,bvw->bctw', x, C_adaptive)


class STGCNBlock(nn.Module):
    """Spatial-Temporal Graph Convolution Block with Relation Prediction."""

    def __init__(self, in_ch, out_ch, A, stride=1, dropout=0.1):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_ch, out_ch, A)
        self.rp = RelationPrediction(in_ch, num_nodes=A.shape[1])
        self.rp_proj = nn.Conv2d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()
        self.relu1 = nn.ReLU(inplace=True)
        self.tcn = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, kernel_size=(9, 1), stride=(stride, 1), padding=(4, 0)),
            nn.BatchNorm2d(out_ch),
        )
        self.relu2 = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(dropout)

        if in_ch != out_ch or stride != 1:
            self.residual = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        out = self.sgcn(x) + self.rp_proj(self.rp(x))
        out = self.relu1(out)
        out = self.tcn(out)
        out = self.relu2(out + res)
        return self.drop(out)


class HPI_GCN_RP(nn.Module):
    """
    HPI-GCN-RP: Human Pose Interaction Graph Convolutional Network
    with Relation Prediction for violence detection.

    Input:  (batch, seq_len=30, features=102)
    Output: (batch,) — logits for BCEWithLogitsLoss
    """

    MODEL_NAME = "HPI-GCN-RP"

    def __init__(self, num_keypoints=17, num_people=2, in_channels=3,
                 num_classes=1, dropout=0.3):
        super().__init__()
        self.num_nodes = num_keypoints * num_people
        self.num_keypoints = num_keypoints
        self.num_people = num_people
        self.in_channels = in_channels

        A = build_coco_adjacency(num_keypoints, num_people)

        self.bn_input = nn.BatchNorm1d(in_channels * self.num_nodes)

        self.layers = nn.ModuleList([
            STGCNBlock(3,   64,  A, stride=1, dropout=0.1),
            STGCNBlock(64,  64,  A, stride=1, dropout=0.1),
            STGCNBlock(64,  128, A, stride=2, dropout=0.1),
            STGCNBlock(128, 128, A, stride=1, dropout=0.1),
            STGCNBlock(128, 256, A, stride=2, dropout=0.1),
            STGCNBlock(256, 256, A, stride=1, dropout=0.1),
        ])

        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        B = x.size(0)
        T = x.size(1)

        # Reshape: (B, T, 102) -> (B, C=3, T, V=34)
        x = x.reshape(B, T, self.num_people, self.num_keypoints, self.in_channels)
        x = x.reshape(B, T, self.num_nodes, self.in_channels)
        x = x.permute(0, 3, 1, 2)  # (B, C, T, V)

        # Input normalization
        x = x.reshape(B, self.in_channels * self.num_nodes, T)
        x = self.bn_input(x)
        x = x.reshape(B, self.in_channels, T, self.num_nodes)

        for layer in self.layers:
            x = layer(x)

        x = x.mean(dim=[2, 3])  # Global avg pool -> (B, 256)
        x = self.drop(x)
        x = self.fc(x)
        return x.squeeze(-1)

## 4. Dataset

In [ ]:
def get_ntu_subject(filename):
    """Extract subject/performer ID from NTU filename: SsssCcccPpppRrrrAaaa."""
    stem = Path(filename).stem
    p_idx = stem.find('P')
    if p_idx >= 0:
        try:
            return int(stem[p_idx+1:p_idx+4])
        except ValueError:
            pass
    return -1


class SkeletonDataset(Dataset):
    """
    Loads pre-extracted skeleton .npy files for HPI-GCN-RP.
    Each .npy file: shape (30, 102) = (frames, features)
    Labels: fight=1, nonfight=0
    """
    def __init__(self, samples: list, augment: bool = False):
        self.samples = samples  # list of (path, label)
        self.augment = augment

        n_fight = sum(1 for _, l in self.samples if l == 1)
        n_nonfight = sum(1 for _, l in self.samples if l == 0)
        print(f"  Dataset: {len(self.samples)} total ({n_fight} fight, {n_nonfight} nonfight)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        seq = np.load(path).astype(np.float32)  # (30, 102)

        if self.augment:
            seq = self._augment(seq)

        return torch.FloatTensor(seq), torch.FloatTensor([label])

    def _augment(self, seq):
        """Data augmentation for skeleton sequences."""
        # 1. Random temporal shift (0-3 frames)
        shift = np.random.randint(0, 4)
        if shift > 0:
            seq = np.roll(seq, shift, axis=0)

        # 2. Horizontal flip (mirror x-coordinates)
        if np.random.random() > 0.5:
            seq_r = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            seq_r[:, :, :, 0] = 1.0 - seq_r[:, :, :, 0]  # flip x
            # Swap left/right keypoints within each person
            swap = [(1,2), (3,4), (5,6), (7,8), (9,10), (11,12), (13,14), (15,16)]
            for l, r in swap:
                seq_r[:, :, [l, r], :] = seq_r[:, :, [r, l], :]
            # Swap person order (P1 <-> P2)
            seq_r[:, [0, 1], :, :] = seq_r[:, [1, 0], :, :]
            seq = seq_r.reshape(SEQUENCE_LEN, -1)

        # 3. Gaussian noise
        noise = np.random.normal(0, 0.01, seq.shape).astype(np.float32)
        seq = seq + noise

        # 4. Joint dropout (GCN-specific: zero out random joints)
        if np.random.random() > 0.8:
            num_drop = np.random.randint(1, 4)
            person = np.random.randint(0, 2)
            joints = np.random.choice(17, num_drop, replace=False)
            seq_r = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            seq_r[:, person, joints, :] = 0
            seq = seq_r.reshape(SEQUENCE_LEN, -1)

        # 5. Random temporal speed variation
        if np.random.random() > 0.7:
            speed = np.random.uniform(0.8, 1.2)
            indices = np.linspace(0, SEQUENCE_LEN - 1, int(SEQUENCE_LEN * speed))
            indices = np.clip(indices, 0, SEQUENCE_LEN - 1).astype(int)
            stretched = seq[indices]
            if len(stretched) != SEQUENCE_LEN:
                new_idx = np.linspace(0, len(stretched) - 1, SEQUENCE_LEN).astype(int)
                seq = stretched[new_idx]

        return seq

## 5. Data Loading (NTU Cross-Subject Split)

In [ ]:
# Collect all samples
all_samples = []
root = Path(SKELETON_DIR)

# Walk through train/ directory (extract_skeletons.py puts everything in train/)
for split_dir in sorted(root.iterdir()):
    if not split_dir.is_dir():
        continue
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label = 1 if 'fight' in class_dir.name and 'non' not in class_dir.name else 0
        for npy_file in sorted(class_dir.glob('*.npy')):
            all_samples.append((str(npy_file), label))

print(f"Total samples found: {len(all_samples)}")
print(f"  Fight: {sum(1 for _, l in all_samples if l == 1)}")
print(f"  Non-fight: {sum(1 for _, l in all_samples if l == 0)}")

# NTU cross-subject split
train_samples = []
val_samples = []

for path, label in all_samples:
    subject = get_ntu_subject(path)
    if subject in TRAIN_SUBJECTS:
        train_samples.append((path, label))
    else:
        val_samples.append((path, label))

# If no NTU filenames detected, fall back to random 80/20 split
if len(val_samples) == 0:
    print("\nNo NTU filenames detected, using random 80/20 split...")
    np.random.seed(42)
    indices = np.random.permutation(len(all_samples))
    split = int(0.8 * len(indices))
    train_samples = [all_samples[i] for i in indices[:split]]
    val_samples = [all_samples[i] for i in indices[split:]]

print(f"\nTrain samples: {len(train_samples)}")
print(f"Val samples: {len(val_samples)}")

# Create datasets
print("\nTraining set:")
train_dataset = SkeletonDataset(train_samples, augment=True)
print("Validation set:")
val_dataset = SkeletonDataset(val_samples, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 6. Model Instantiation

In [ ]:
model = HPI_GCN_RP(
    num_keypoints=NUM_KEYPOINTS,
    num_people=NUM_PEOPLE,
    in_channels=IN_CHANNELS,
    num_classes=1,
    dropout=0.3,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"HPI-GCN-RP: {total_params:,} parameters ({trainable:,} trainable)")
print()

# Quick shape test
dummy = torch.randn(2, SEQUENCE_LEN, FEATURE_DIM).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"Input:  {dummy.shape}")
print(f"Output: {out.shape}")
print(f"Values: {out}")

## 7. Loss, Optimizer, Scheduler

In [ ]:
# Class weight for imbalance
fight_count = sum(1 for _, l in train_samples if l == 1)
nonfight_count = sum(1 for _, l in train_samples if l == 0)
if fight_count > 0:
    pos_weight = torch.tensor([nonfight_count / fight_count]).to(device)
else:
    pos_weight = torch.tensor([1.0]).to(device)
print(f"Class ratio (nonfight/fight): {pos_weight.item():.2f}")
print(f"Label smoothing: {LABEL_SMOOTHING}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

print(f"Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingWarmRestarts (T_0=10, T_mult=2)")

## 8. Training Loop

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'lr': []}
best_val_acc = 0.0
best_val_f1 = 0.0
best_epoch = 0


def smooth_labels(labels, smoothing=0.05):
    """Label smoothing: 0 -> smoothing, 1 -> 1-smoothing."""
    return labels * (1 - smoothing) + smoothing * 0.5


def train_epoch(model, loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    # Warmup: linearly increase LR for first WARMUP_EPOCHS
    if epoch <= WARMUP_EPOCHS:
        warmup_lr = LEARNING_RATE * epoch / WARMUP_EPOCHS
        for param_group in optimizer.param_groups:
            param_group['lr'] = warmup_lr

    for sequences, labels in loader:
        sequences = sequences.to(device)     # (B, 30, 102)
        labels = labels.squeeze().to(device)  # (B,)

        # Apply label smoothing
        smooth = smooth_labels(labels, LABEL_SMOOTHING)

        optimizer.zero_grad()
        logits = model(sequences)
        loss = criterion(logits, smooth)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += len(labels)

    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for sequences, labels in loader:
        sequences = sequences.to(device)
        labels = labels.squeeze().to(device)

        logits = model(sequences)
        loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += len(labels)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds)
    return total_loss / total, correct / total, f1, all_preds, all_labels

In [ ]:
print(f"Training HPI-GCN-RP for {EPOCHS} epochs on {device}...")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | {'Val F1':>7} | {'LR':>10}")
print("-" * 80)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, epoch)
    val_loss, val_acc, val_f1, _, _ = eval_epoch(model, val_loader, criterion)

    if epoch > WARMUP_EPOCHS:
        scheduler.step()

    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(lr)

    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_f1 = val_f1
        best_epoch = epoch
        marker = ' ** BEST'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_f1': val_f1,
            'val_loss': val_loss,
            'model_name': 'HPI-GCN-RP',
            'num_keypoints': NUM_KEYPOINTS,
            'num_people': NUM_PEOPLE,
            'in_channels': IN_CHANNELS,
            'sequence_len': SEQUENCE_LEN,
        }, 'best_hpi_gcn_rp.pt')

    print(f"{epoch:>5} | {train_loss:>10.4f} | {train_acc:>8.1%} | {val_loss:>10.4f} | {val_acc:>8.1%} | {val_f1:>6.3f} | {lr:>10.6f}{marker}")

print(f"\nBest: epoch {best_epoch}, val_acc = {best_val_acc:.1%}, val_f1 = {best_val_f1:.3f}")

## 9. Evaluation

In [ ]:
# Load best model
checkpoint = torch.load('best_hpi_gcn_rp.pt')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")
print(f"Val accuracy: {checkpoint['val_acc']:.1%}")
print(f"Val F1: {checkpoint['val_f1']:.3f}")

# Final evaluation
val_loss, val_acc, val_f1, all_preds, all_labels = eval_epoch(model, val_loader, criterion)

print(f"\nFinal Val Accuracy: {val_acc:.1%}")
print(f"Final Val F1: {val_f1:.3f}")
print(f"Final Val Loss: {val_loss:.4f}")

# Classification report
print("\n" + classification_report(
    all_labels, all_preds,
    target_names=['Non-Fight', 'Fight'],
    digits=3
))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0].set_title('Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', linewidth=2)
axes[1].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[1].set_title('Accuracy', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['val_f1'], label='Val F1', linewidth=2, color='green')
axes[2].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[2].set_title('Val F1 Score', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('HPI-GCN-RP Training Curves', fontsize=16)
plt.tight_layout()
plt.savefig('hpi_gcn_rp_training_curves.png', dpi=150)
plt.show()
print("Saved: hpi_gcn_rp_training_curves.png")

## 10. Export Model

In [ ]:
# Export for AI Guardian integration
export_path = "hpi_gcn_rp.pt"

torch.save({
    'model_state_dict': model.state_dict(),
    'model_name': 'HPI-GCN-RP',
    'num_keypoints': NUM_KEYPOINTS,
    'num_people': NUM_PEOPLE,
    'in_channels': IN_CHANNELS,
    'sequence_len': SEQUENCE_LEN,
    'val_accuracy': best_val_acc,
    'val_f1': best_val_f1,
    'epoch': best_epoch,
}, export_path)

file_size = os.path.getsize(export_path) / 1024
print(f"Exported: {export_path} ({file_size:.0f} KB)")
print(f"Val accuracy: {best_val_acc:.1%}")
print(f"Val F1: {best_val_f1:.3f}")
print(f"\n{'='*50}")
print(f"DOWNLOAD THIS FILE")
print(f"Rename to: best_model.pth")
print(f"Put in: AiGuardian/models/best_model.pth")
print(f"classifier.py will auto-detect HPI-GCN-RP and load it.")
print(f"{'='*50}")

## 11. Inference Speed Test

In [ ]:
import time

model.eval()

# GPU inference
dummy = torch.randn(1, SEQUENCE_LEN, FEATURE_DIM).to(device)
# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy)
if device.type == 'cuda':
    torch.cuda.synchronize()

# Measure
times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        out = model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    times.append((time.perf_counter() - start) * 1000)

print(f"GPU inference ({device}):")
print(f"  Mean: {np.mean(times):.2f} ms")
print(f"  Std:  {np.std(times):.2f} ms")
print(f"  Min:  {np.min(times):.2f} ms")
print(f"  Max:  {np.max(times):.2f} ms")

# CPU inference (important for deployment)
model_cpu = model.cpu()
dummy_cpu = dummy.cpu()
# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model_cpu(dummy_cpu)

times_cpu = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        out = model_cpu(dummy_cpu)
    times_cpu.append((time.perf_counter() - start) * 1000)

print(f"\nCPU inference:")
print(f"  Mean: {np.mean(times_cpu):.2f} ms")
print(f"  Std:  {np.std(times_cpu):.2f} ms")
print(f"  Target: <10 ms (for real-time at 30 FPS)")

# Move back to GPU
model = model.to(device)

## 12. Sample Predictions

In [ ]:
model.eval()

print("Sample predictions:")
print(f"{'#':>3} | {'True':>6} | {'Pred':>6} | {'Score':>6} | {'Correct':>7}")
print("-" * 42)

n_correct = 0
n_total = min(30, len(val_dataset))

for i in range(n_total):
    seq, label = val_dataset[i]
    seq = seq.unsqueeze(0).to(device)

    with torch.no_grad():
        logit = model(seq)
        score = torch.sigmoid(logit).item()

    true_label = 'Fight' if label.item() > 0.5 else 'Normal'
    pred_label = 'Fight' if score > 0.5 else 'Normal'
    correct = true_label == pred_label
    n_correct += int(correct)
    mark = '  OK' if correct else '  MISS'

    print(f"{i+1:>3} | {true_label:>6} | {pred_label:>6} | {score:>5.3f} | {mark}")

print(f"\nSample accuracy: {n_correct}/{n_total} = {n_correct/n_total:.1%}")